<a href="https://colab.research.google.com/github/trainocate-japan/openai_api_app/blob/main/chapter3/%E7%AC%AC3%E7%AB%A0%20%E7%94%BB%E5%83%8F%E7%94%9F%E6%88%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 画像生成

In [ ]:
# パッケージインストール
# !pip install openai
# !pip install tiktoken
# !pip install openai
# streamlit関連パッケージのインストール
!pip install streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!which cloudflared && cloudflared --version

In [ ]:
from openai import OpenAI
client = OpenAI(api_key="your api key")

In [ ]:
response = client.images.generate(
  model="dall-e-2",
  prompt="a white siamese cat",
  size="512x512",
  n=1
)

image_url = response.data[0].url
print(image_url)

In [ ]:
from PIL import Image
import requests
import io

image = Image.open(io.BytesIO(requests.get(image_url).content))

image.save("generated_image.png")

image

In [ ]:
# app.pyの中身を実装
%%writefile app.py
from openai import OpenAI
import streamlit as st
from PIL import Image
import requests
import io

client = OpenAI(api_key="your api key")

st.title("画像を生成するボット")
user_input = st.text_input("生成したい画像のキーワードを入力してください:")

if st.button("送信"):
    response = client.images.generate(
        model="dall-e-2",
        prompt=user_input,
        size="512x512",
        n=1
    )
    st.text_area("ボットの応答", "画像を生成しました")
    image = Image.open(io.BytesIO(requests.get(response.data[0].url).content))
    st.image(image, caption='Created by AI',use_column_width=True)

In [ ]:
# streamlitのrunコマンドでapp.pyを立ち上げ、localtunnelを用いてアプリ公開
# !streamlit run app.py & sleep 3 && npx localtunnel --port 8501
!streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.fileWatcherType none \
  > /tmp/st.log 2>&1 &
!for i in {1..60}; do curl -fsS http://localhost:8501/healthz && echo "Streamlit is up" && break || sleep 1; done

# トンネル起動（ログにURLが出る）
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate > /tmp/cf.log 2>&1 &

# URLがログに出るまで最大60秒待って抽出
!for i in {1..60}; do \
  URL=$(grep -o "https://[0-9a-z.-]*trycloudflare.com" -m 1 /tmp/cf.log); \
  if [ -n "$URL" ]; then echo "PUBLIC URL: $URL"; break; fi; \
  sleep 1; \
done

# 画像編集

In [ ]:
from PIL import Image
import requests
import io
from openai import OpenAI

client = OpenAI(api_key="your api key")

In [ ]:
image_name = "8044547720_628e930721_z.png"

image = Image.open(image_name)
image

In [ ]:
mask_name = "8044547720_628e930721_z_mask_1.png"

image = Image.open(mask_name)
image

In [ ]:
# メソッドを呼び出し、応答を得る
response = client.images.edit(
    model="dall-e-2",
    image=open(image_name, "rb"),
    mask=open(mask_name, "rb"),
    prompt="羊を消して",
    n=1,
    size="512x512"
)

image_url = response.data[0].url

image = Image.open(io.BytesIO(requests.get(image_url).content))
image

In [ ]:
mask_name = "8044547720_628e930721_z_mask_2.png"

image = Image.open(mask_name)
image

In [ ]:
# メソッドを呼び出し、応答を得る
response = client.images.edit(
    model="dall-e-2",
    image=open(image_name, "rb"),
    mask=open(mask_name, "rb"),
    prompt="犬を増やして",
    n=1,
    size="512x512"
)

image_url = response.data[0].url

image = Image.open(io.BytesIO(requests.get(image_url).content))
image